# 198. House Robber
**Difficulty:** 🟡 Medium · **Topic:** Dynamic Programming · **LeetCode:** https://leetcode.com/problems/house-robber/

## 💡 Concepts

**Core concept(s):** 1-D DP — at each house choose the better of **skip it** or **rob it + best up to two houses back**.

**Why it applies here:** You can't rob two adjacent houses. So the best loot up to house `i` is either the best up to `i-1` (skip `i`) or `nums[i]` plus the best up to `i-2` (rob `i`). Only the last two results are ever needed.

**Key intuition:** Best so far = max(skip this house, rob this house + best from two houses ago).

---

### 📚 What is Dynamic Programming (DP)?
**DP** solves a big problem by solving smaller **overlapping** subproblems once and reusing the answers. Two styles: **memoization** (recursion that caches results) and **tabulation** (fill a table from the smallest cases up).
- **Why it's fast:** it turns exponential re-computation into a single sweep over the subproblems.
- **In Python:** a `dict`/list cache, or a `dp` list/2-D table.

### 📚 Subproblems & Recurrence
The heart of DP is a **recurrence**: the answer for a state written in terms of smaller states (e.g. `dp[i] = dp[i-1] + dp[i-2]`). Find the recurrence and the base cases, and the code writes itself.

---

**Prerequisite knowledge:**
- The skip/take recurrence.
- Rolling two variables.

## 📝 Problem

Given house values in a row, maximize the loot without robbing two adjacent houses.

**Example**
```
[2,7,9,3,1] -> 12   (2 + 9 + 1)
```

> Two approaches (plus a brute force): O(n) array DP and O(1)-space rolling.

### Approach 1 — Brute Recursion (worst)

**Idea:** At each house, try skipping or robbing (then jump two ahead); take the max.

**Time:** `O(2^n)`. **Space:** `O(n)`.

In [ ]:
def rob_brute(nums):
    def dfs(i):                            # most money robbing from house i onward
        if i >= len(nums):
            return 0
        # Either skip house i, or rob it (+ its money) and jump two houses ahead.
        return max(dfs(i + 1), nums[i] + dfs(i + 2))
    return dfs(0)

### Approach 2 — Array DP (better)

**Idea:** `dp[i]` = best loot considering the first `i` houses.

**Time:** `O(n)`. **Space:** `O(n)`.

In [ ]:
def rob_array(nums):
    if not nums:
        return 0
    n = len(nums)
    dp = [0] * (n + 1)                      # dp[i] = best loot from the first i houses
    dp[1] = nums[0]
    for i in range(2, n + 1):
        # Best is either skip house i-1 (dp[i-1]) or rob it plus dp[i-2].
        dp[i] = max(dp[i - 1], dp[i - 2] + nums[i - 1])
    return dp[n]

### Approach 3 — Rolling Two Variables (optimal)

**Idea:** Keep only the best-up-to-previous and best-up-to-two-back.

**Time:** `O(n)`. **Space:** `O(1)`.

In [ ]:
def rob_opt(nums):
    prev, cur = 0, 0                        # best loot up to two houses back / one house back
    for x in nums:
        prev, cur = cur, max(cur, prev + x)  # skip this house, or rob it (x + prev)
    return cur

In [ ]:
# Correctness check
tests = [([2,7,9,3,1],12), ([1,2,3,1],4), ([5],5), ([],0), ([2,1,1,2],4)]
for nums, exp in tests:
    a, b, c = rob_brute(nums), rob_array(nums), rob_opt(nums)
    print(f"{nums} -> brute={a}, array={b}, opt={c} | expected={exp}")
    assert a == b == c == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |

Inputs are shaped to force the worst case. (Exponential brute-force versions are shown in the code but omitted from timing where they would blow up — noted per notebook.)

*(Brute force omitted; both efficient approaches are linear.)*

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return ([(i * 37) % 100 for i in range(n)],)
solutions = {
    "array DP O(n) space O(n)": rob_array,
    "rolling  O(n) space O(1)": rob_opt,
}
sizes = [20000, 40000, 80000, 160000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Take-or-skip DP:** classic 1-D pattern with a two-step-back dependency.
- **Roll the array away:** only two prior values matter → O(1) space.
- **Signal:** "max sum with no two adjacent", "choose items with a spacing rule".
- **Related problems:** House Robber II, Delete and Earn, Max Sum of Non-Adjacent, Paint House.
- **Common pitfalls:** (1) seeding the base cases wrong; (2) trying to be greedy (fails).